# EU Loneliness Survey Analysis
 
## About the Dataset
 
| Field | Details |
|-------|---------|
| **Source** | EU Loneliness Survey 2022 |
| **Conducted by** | Joint Research Centre (JRC), European Commission |
| **Data Collection** | VVA Market Research via Cint platform |
| **Sample Size** | 25,646 respondents |
| **Coverage** | All 27 EU Member States |
| **Target Population** | General population aged 16 and above |
| **Method** | Online survey (November–December 2022) |
 
> Data available at: https://data.jrc.ec.europa.eu <br> 
https://data.jrc.ec.europa.eu/dataset/82e60986-9987-4610-ab4a-84f0f5a9193b

 
---
 
## Project Objective
 
To explore the patterns and factors associated with loneliness across Europe using survey data.  
Key questions explored:
- Who is the loneliest? (by age, gender, work status)
- What factors increase loneliness? (health, life events, pets)
- Does social media usage affect loneliness?
- How do demographics like occupation and sexual orientation relate to loneliness?
 
---

## Step 1 — Import & Initial Exploration
 
Loading the dataset and checking its shape, columns, and data types.

In [1]:
import pandas as pd

loneliness = pd.read_excel('final_dataset.xlsx')
print(loneliness.shape)
print(loneliness.columns.tolist())
loneliness.head()

(25646, 38)
['country', 'date_birth_year', 'gender', 'relationship', 'sex_orientation', 'income_decile', 'has_pet', 'work_status', 'occupation', 'worklife_balance_rev', 'telework', 'loneliness_djg_average score', 'loneliness_ucla_score', 'overall loneliness index', 'relationship_need', 'loneliness_direct', 'life_event_happen', 'work_loneliness_score', 'sm_helps_score', 'sm_purpose2_a', 'sm_purpose2_b', 'sm_purpose2_c', 'sm_purpose2_d', 'sm_time_a', 'sm_time_b', 'sm_time_c', 'sm_time_d', 'child_lived_parents', 'child_friends', 'religion_now', 'health_general', 'health_condition', 'weight__open', 'height__open', 'smoking', 'fruits', 'exercise', 'age']


,country,date_birth_year,gender,relationship,sex_orientation,income_decile,has_pet,work_status,occupation,worklife_balance_rev,...,child_friends,religion_now,health_general,health_condition,weight__open,height__open,smoking,fruits,exercise,age
0,1,1990,1.0,1.0,1.0,7.0,1,1,5.0,4.0,...,4.0,1.0,3.0,2.0,85.0,186.0,3.0,2.0,5.0,32.0
1,1,1966,1.0,1.0,1.0,5.0,0,1,5.0,3.0,...,1.0,2.0,4.0,1.0,85.0,188.0,2.0,2.0,2.0,56.0
2,1,1977,1.0,1.0,1.0,3.0,0,1,7.0,3.0,...,3.0,1.0,3.0,NaN,88.0,178.0,3.0,1.0,4.0,45.0
3,1,1987,2.0,2.0,1.0,6.0,1,1,3.0,8.0,...,3.0,1.0,3.0,1.0,70.0,160.0,2.0,1.0,4.0,35.0
4,1,1979,1.0,3.0,1.0,9.0,1,1,6.0,1.0,...,1.0,2.0,2.0,1.0,63.0,169.0,1.0,1.0,8.0,43.0


In [2]:
loneliness.dtypes

country                           int64
date_birth_year                   int64
gender                          float64
relationship                    float64
sex_orientation                 float64
income_decile                   float64
has_pet                           int64
work_status                       int64
occupation                      float64
worklife_balance_rev            float64
telework                        float64
loneliness_djg_average score    float64
loneliness_ucla_score           float64
overall loneliness index        float64
relationship_need               float64
loneliness_direct               float64
life_event_happen                 int64
work_loneliness_score           float64
sm_helps_score                  float64
sm_purpose2_a                   float64
sm_purpose2_b                   float64
sm_purpose2_c                   float64
sm_purpose2_d                   float64
sm_time_a                       float64
sm_time_b                       float64


## Step 2 — Missing Value Analysis
 
Checking for missing values across all columns.  
Columns with missing data are identified and their missing percentage is calculated.
 
> **Decision:** Missing values were not dropped or imputed at this stage.  
> Power BI handles nulls by excluding them from aggregations automatically.

In [3]:
missing_values=loneliness.isnull().sum()
missing_pct = ((loneliness.isnull().sum() / len(loneliness)) * 100)
missing_df = pd.DataFrame({
    'Missing Values': missing_values,
    'Missing Percentage': missing_pct.round(2)
})

missing_df=missing_df[missing_df['Missing Values']>0]
missing_df.sort_values('Missing Percentage', ascending=False)

,Missing Values,Missing Percentage
occupation,9390,36.61
work_loneliness_score,9130,35.60
telework,8987,35.04
worklife_balance_rev,8987,35.04
health_condition,3518,13.72
weight__open,2610,10.18
height__open,2229,8.69
income_decile,1956,7.63
sm_helps_score,1899,7.40
exercise,1634,6.37


In [4]:
cols_to_check = [
    'overall loneliness index',
    'loneliness_djg_average score',
    'loneliness_ucla_score',
    'loneliness_direct',
    'sm_time_a',        # social network sites time
    'sm_time_b',        # instant messaging time
    'sm_time_c',        # gaming time
    'sm_purpose2_a',    # use SM to meet new people
    'sm_purpose2_b',    # use SM for shared interests
    'sm_purpose2_c',    # more confident online than face to face
    'sm_purpose2_d',    # SM replaces face to face contact
    'age',
    'gender',
    'country'
]

loneliness[cols_to_check].describe()

,overall loneliness index,loneliness_djg_average score,loneliness_ucla_score,loneliness_direct,sm_time_a,sm_time_b,sm_time_c,sm_purpose2_a,sm_purpose2_b,sm_purpose2_c,sm_purpose2_d,age,gender,country
count,25434.000000,25644.000000,25436.000000,24611.000000,25315.000000,25283.000000,25254.000000,24580.000000,24544.000000,24539.000000,24548.000000,25634.000000,25566.000000,25646.000000
mean,1.966850,2.252169,1.672433,3.839137,3.852459,3.442432,2.722579,4.626729,5.951190,4.678186,4.648240,44.322462,1.527145,13.975630
std,0.243279,0.534104,0.619998,1.120061,1.778813,1.660261,1.916591,2.853719,2.707452,2.854727,2.833748,15.197626,0.508125,7.864067
min,0.500000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,15.000000,1.000000,1.000000
25%,1.833333,2.000000,1.000000,3.000000,3.000000,2.000000,1.000000,2.000000,4.000000,2.000000,2.000000,32.000000,1.000000,7.000000
50%,2.000000,2.333333,1.666667,4.000000,4.000000,3.000000,2.000000,5.000000,6.000000,5.000000,5.000000,44.000000,2.000000,14.000000
75%,2.083333,2.666667,2.000000,5.000000,5.000000,4.000000,4.000000,7.000000,8.000000,7.000000,7.000000,55.000000,2.000000,21.000000
max,3.000000,3.000000,3.000000,5.000000,8.000000,8.000000,8.000000,10.000000,10.000000,10.000000,10.000000,91.000000,3.000000,27.000000


In [5]:
loneliness.dtypes

country                           int64
date_birth_year                   int64
gender                          float64
relationship                    float64
sex_orientation                 float64
income_decile                   float64
has_pet                           int64
work_status                       int64
occupation                      float64
worklife_balance_rev            float64
telework                        float64
loneliness_djg_average score    float64
loneliness_ucla_score           float64
overall loneliness index        float64
relationship_need               float64
loneliness_direct               float64
life_event_happen                 int64
work_loneliness_score           float64
sm_helps_score                  float64
sm_purpose2_a                   float64
sm_purpose2_b                   float64
sm_purpose2_c                   float64
sm_purpose2_d                   float64
sm_time_a                       float64
sm_time_b                       float64


In [6]:
for col in ['child_friends', 'child_lived_parents']:
    print(f"\n{col}:")
    print(loneliness[col].value_counts(dropna=False))


child_friends:
child_friends
2.0    9924
1.0    7793
3.0    4747
4.0    2079
5.0     721
NaN     382
Name: count, dtype: int64

child_lived_parents:
child_lived_parents
1.0    19963
2.0     5190
NaN      493
Name: count, dtype: int64


## Step 3 — Label Encoding
 
Raw survey responses use numeric codes (e.g. 1, 2, 3).  
We map these to meaningful labels for readability in analysis and dashboards.
 
Labels created for:
- `gender` → Male, Female, Other
- `relationship` → Single, Married, In a relationship, etc.
- `sex_orientation` → Heterosexual, Bisexual, Lesbian or gay, Other
- `work_status` → Employee, Studying, Unemployed, Retired, etc.
- `occupation` → Professional, Unskilled, Farm worker, etc.
- `has_pet` → Has Pet, No Pet
- `telework` → Work from home, Work from Office, Somewhere Else
- `health_condition` → Yes, No
- `smoking` → Heavy Smoker, Light Smoker, Non Smoker
- `fruits` → Yes, No
- `life_event_happen` → Yes, No
- `country` → Austria, Belgium, Bulgaria... (all 27 EU countries)
 
---

In [7]:
gender_map={
    1.0:'Male',
    2.0:'Female',
    3.0:'Other'}
loneliness['gender_label']=loneliness['gender'].map(gender_map)

In [8]:
loneliness['gender_label']

0          Male
1          Male
2          Male
3        Female
4          Male
          ...  
25641      Male
25642      Male
25643      Male
25644       NaN
25645     Other
Name: gender_label, Length: 25646, dtype: object

In [9]:
print(loneliness['gender_label'].value_counts(dropna=False))

gender_label
Female    13249
Male      12203
Other       114
NaN          80
Name: count, dtype: int64


In [10]:
relationship_map={
    1.0:'Single',
    2.0:'In a relationship',
    3.0:'Married',
    4.0:'Separated or Divorces',
    5.0:'Widowed'
}
loneliness['relationship_label']=loneliness['relationship'].map(relationship_map)
loneliness['relationship_label'].value_counts(dropna=False)

relationship_label
Married                  13669
Single                    5530
In a relationship         3714
Separated or Divorces     1872
Widowed                    679
NaN                        182
Name: count, dtype: int64

In [11]:
sex_orientation_label={
    1.0:'Hetrosexual/straight',
    2.0:'Lesbian or gay',
    3.0: 'Bisexual',
    4.0:'Other'
}
loneliness['sex_orientation_label']=loneliness['sex_orientation'].map(sex_orientation_label)
loneliness['sex_orientation_label'].value_counts(dropna=False).sort_values(ascending=False)


sex_orientation_label
Hetrosexual/straight    22602
NaN                      1143
Bisexual                 1007
Lesbian or gay            632
Other                     262
Name: count, dtype: int64

In [12]:
work_label={
    1.0:'Employee/Self-Employed',
    2.0:'Studying',
    3.0:'Unemployed-Actively looking',
    4.0:'Unemployed-Not actively looking',
    5.0:'Sick or Disabled',
    6.0:'Retired',
    7.0:'Housework',
    8.0:'Other'
}
loneliness['work_label']=loneliness['work_status'].map(work_label)
loneliness['work_label'].value_counts(dropna=False).sort_values(ascending=False)

work_label
Employee/Self-Employed             16659
Retired                             3659
Studying                            1555
Unemployed-Actively looking         1281
Housework                            805
Unemployed-Not actively looking      577
Other                                556
Sick or Disabled                     554
Name: count, dtype: int64

In [13]:
occupation_label={
    1.0:'Professional/Technical(Doctor/Engineer/Teacher)',
    2.0:'High Admin(Bank/Government/Executives)',
    3.0:'Clerical(Secretary/Clerk/Bookkeeper)',
    4.0:'Sales(Shopowner/Insurance/Salesmanager)',
    5.0:'Service(Police/caretaker/Waiter/Armed)',
    6.0:'Skilled(Mechanic/Printer/Electrician)',
    7.0:'Semi-Skilled(Busdriver/Carpenter/Baker)',
    8.0:'Unskilled(labourer/porter/factoryworker)',
    9.0:'Farm(Farmer/Fisherman/Tractordriver)'
}

loneliness['occupation_label']=loneliness['occupation'].map(occupation_label)
loneliness['occupation_label'].value_counts(dropna=False).sort_values(ascending=False)

occupation_label
NaN                                                9390
Professional/Technical(Doctor/Engineer/Teacher)    4297
Clerical(Secretary/Clerk/Bookkeeper)               4046
Skilled(Mechanic/Printer/Electrician)              1863
High Admin(Bank/Government/Executives)             1538
Sales(Shopowner/Insurance/Salesmanager)            1523
Service(Police/caretaker/Waiter/Armed)             1492
Unskilled(labourer/porter/factoryworker)            717
Semi-Skilled(Busdriver/Carpenter/Baker)             645
Farm(Farmer/Fisherman/Tractordriver)                135
Name: count, dtype: int64

In [14]:
pet_label={
    1:'Has Pet',
    0:'No Pet'
}
loneliness['pet_label']=loneliness['has_pet'].map(pet_label)
loneliness['pet_label'].value_counts(dropna=False)

pet_label
Has Pet    16845
No Pet      8801
Name: count, dtype: int64

In [15]:
child_lived_parents_label={
    1.0:'Yes',
    2.0:'No'
}
loneliness['child_lived_parents_label']=loneliness['child_lived_parents'].map(child_lived_parents_label)
loneliness['child_lived_parents_label'].value_counts(dropna=False)

child_lived_parents_label
Yes    19963
No      5190
NaN      493
Name: count, dtype: int64

In [16]:
remote_work_label={
    1.0:'Work from home',
    2.0:'Work from Office',
    3.0:'Somewhere Else'
}
loneliness['remote_work_label']=loneliness['telework'].map(remote_work_label)
loneliness['remote_work_label'].value_counts(dropna=False)

remote_work_label
Work from Office    11312
NaN                  8987
Work from home       3892
Somewhere Else       1455
Name: count, dtype: int64

In [17]:
health_condition_label={
    1.0:'Yes',
    2.0:'No'
}
loneliness['health_condition_label']=loneliness['health_condition'].map(health_condition_label)
loneliness['health_condition_label'].value_counts(dropna=False)

health_condition_label
No     14220
Yes     7908
NaN     3518
Name: count, dtype: int64

In [18]:
smoking_label={
    1.0:'Heavy Smoker',
    2.0:'light Smoker',
    3.0:'Non Smoker'
}
loneliness['Smoker_label']=loneliness['smoking'].map(smoking_label)
loneliness['Smoker_label'].value_counts(dropna=False)

Smoker_label
Non Smoker      16128
Heavy Smoker     5307
light Smoker     3825
NaN               386
Name: count, dtype: int64

In [19]:
fruit_label={
    1.0:'Yes',
    2.0:'No'
}
loneliness['eat_fruits_label']=loneliness['fruits'].map(fruit_label)
loneliness['eat_fruits_label'].value_counts(dropna=False)

eat_fruits_label
Yes    12868
No     12270
NaN      508
Name: count, dtype: int64

In [20]:
life_event_label={
    1:'Yes',
    0:'No'
}
loneliness['life_event_label']=loneliness['life_event_happen'].map(life_event_label)
loneliness['life_event_label'].value_counts(dropna=False)

life_event_label
Yes    16655
No      8991
Name: count, dtype: int64

In [21]:
country_label = {
    1: 'Austria',
    2: 'Belgium',
    3: 'Bulgaria',
    4: 'Croatia',
    5: 'Cyprus',
    6: 'Czechia',
    7: 'Denmark',
    8: 'Estonia',
    9: 'Finland',
    10: 'France',
    11: 'Germany',
    12: 'Greece',
    13: 'Hungary',
    14: 'Ireland',
    15: 'Italy',
    16: 'Latvia',
    17: 'Lithuania',
    18: 'Luxembourg',
    19: 'Malta',
    20: 'Netherlands',
    21: 'Poland',
    22: 'Portugal',
    23: 'Romania',
    24: 'Slovakia',
    25: 'Slovenia',
    26: 'Spain',
    27: 'Sweden'
}

loneliness['country_label'] = loneliness['country'].map(country_label)

In [22]:
loneliness['country_label'].value_counts(dropna=False)

country_label
Germany        1105
Lithuania      1012
Ireland        1010
Spain          1009
Latvia         1009
Croatia        1009
Estonia        1009
Finland        1008
Denmark        1008
Austria        1008
Romania        1008
Netherlands    1008
Slovenia       1008
Greece         1008
Sweden         1007
Belgium        1004
Portugal       1004
Slovakia       1003
Hungary        1003
Czechia        1002
Poland         1001
Bulgaria       1001
France         1000
Italy          1000
Malta           529
Cyprus          503
Luxembourg      370
Name: count, dtype: int64

## Step 4 — Age Binning (Generations)
 
Age is grouped into generations for clearer analysis:
 
| Generation | Age Range |
|------------|-----------|
| Gen Z | 15–27 |
| Millennial | 28–43 |
| Gen X | 44–59 |
| Boomer+ | 60–91 |

In [23]:
bins = [15, 27, 43, 59, 91]
labels = ['Gen Z', 'Millennial', 'Gen X', 'Boomer+']
loneliness['generation'] = pd.cut(loneliness['age'], bins=bins, labels=labels)
loneliness.groupby('generation')['overall loneliness index'].mean()

C:\Users\Kanya\AppData\Local\Temp\ipykernel_32348\3169784830.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  loneliness.groupby('generation')['overall loneliness index'].mean()


generation
Gen Z         2.009158
Millennial    1.976522
Gen X         1.947536
Boomer+       1.945868
Name: overall loneliness index, dtype: float64

## Step 5 — Income Binning
 
Income decile (1–10) is grouped into three ranges:
 
| Income Range | Decile |
|-------------|--------|
| Low income | 1–3 |
| Middle income | 4–7 |
| High income | 8–10 |
 
---

In [24]:
print(loneliness['income_decile'].unique())
print(loneliness['income_decile'].describe())

[ 7.  5.  3.  6.  9.  2.  1.  8.  4. nan 10.]
count    23690.000000
mean         5.944027
std          2.689025
min          1.000000
25%          4.000000
50%          6.000000
75%          8.000000
max         10.000000
Name: income_decile, dtype: float64


In [25]:
bins = [1, 3, 7, 10]
labels = ['Low income', 'Middle income', 'High income']
loneliness['income_range'] = pd.cut(loneliness['income_decile'], bins=bins, labels=labels, include_lowest=True)
loneliness.groupby('income_range')['overall loneliness index'].mean()

C:\Users\Kanya\AppData\Local\Temp\ipykernel_32348\2530897172.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  loneliness.groupby('income_range')['overall loneliness index'].mean()


income_range
Low income       1.990876
Middle income    1.972692
High income      1.949647
Name: overall loneliness index, dtype: float64

## Step 6 — Categorizing Social Media Avoidance

The variable **sm_purpose2_d** is grouped into levels to represent the tendency to use social media to avoid human interaction:

| Category | Score Range |
|----------|------------|
| Low      | 0–3        |
| Medium   | 3–6        |
| High     | 6–10       |

In [26]:
bins=[0,3,6,10]
labels=['Low','Medium','High']
loneliness['sm_to_avoid_humans_connection']=pd.cut(loneliness['sm_purpose2_d'], bins=bins,labels=labels, include_lowest=True)

## Step 7 — Duplicate Check
 
Checking for duplicate rows in the dataset.
 
---

In [27]:
loneliness.duplicated().sum()

np.int64(0)

## Step 8 — Descriptive Statistics
 
Summary statistics for key variables:
- `overall loneliness index`
- `loneliness_direct`
- `sm_time_a` (social media usage time)
- `age`
 
---

In [28]:
loneliness[['overall loneliness index','loneliness_direct']].describe()

,overall loneliness index,loneliness_direct
count,25434.000000,24611.000000
mean,1.966850,3.839137
std,0.243279,1.120061
min,0.500000,1.000000
25%,1.833333,3.000000
50%,2.000000,4.000000
75%,2.083333,5.000000
max,3.000000,5.000000


In [29]:
loneliness['sm_time_a'].describe()

count    25315.000000
mean         3.852459
std          1.778813
min          1.000000
25%          3.000000
50%          4.000000
75%          5.000000
max          8.000000
Name: sm_time_a, dtype: float64

## Step 9 — Correlation Analysis
 
### Social Media Time vs Loneliness
Exploring whether more time on social media is associated with higher loneliness.
 
### Social Media Purpose vs Loneliness
Checking which SM usage purpose correlates most with loneliness:
- `sm_purpose2_a` — Use SM to meet new people
- `sm_purpose2_b` — Use SM for shared interests
- `sm_purpose2_c` — Feel more confident online than face-to-face
- `sm_purpose2_d` — Use SM to replace face-to-face contact

### sm_helps_score
Correlation between `sm_helps_score` and loneliness index = **~-0.01**  
This near-zero correlation suggests that whether a person feels social media helps them has **no meaningful impact** on their loneliness levels.  
This variable was **excluded from the final dashboard**.
 
---

In [30]:
corr_value=loneliness['overall loneliness index'].corr(loneliness['loneliness_direct'])
corr_value

np.float64(-0.2084436423692604)

In [31]:
corr_value=loneliness['sm_time_a'].corr(loneliness['overall loneliness index'])
corr_value

np.float64(0.08452641558014606)

In [32]:
sm_purpose=['sm_purpose2_a','sm_purpose2_b','sm_purpose2_c','sm_purpose2_d']

corr_sm_purpose=loneliness[sm_purpose].corrwith(loneliness['overall loneliness index'])
corr_sm_purpose

sm_purpose2_a    0.067131
sm_purpose2_b    0.056292
sm_purpose2_c    0.075485
sm_purpose2_d    0.075794
dtype: float64

In [33]:
loneliness['work_loneliness_score'].corr(loneliness['overall loneliness index'])

np.float64(0.07441947022479456)

In [34]:
loneliness['worklife_balance_rev'].corr(loneliness['overall loneliness index'])

np.float64(0.02604322584955874)

In [35]:
print(loneliness['sm_helps_score'].value_counts().sort_index())
corr = loneliness['sm_helps_score'].corr(loneliness['overall loneliness index'])
print(f"Correlation: {corr}")

sm_helps_score
1.0    14316
1.5     4655
2.0     4776
Name: count, dtype: int64
Correlation: -0.015491717238907092


## Step 10 — Group Analysis
 
Comparing average loneliness index across key groups:
 
| Variable | Finding |
|----------|---------|
| Work status | Students are loneliest (2.00), Retired are least lonely |
| Pet ownership | Has Pet slightly lonelier than No Pet |
| Health condition | Yes (1.98) vs No (1.95) |
| Life events | Yes (1.98) vs No (1.94) — 2.06% increase |
| Generation | Gen Z (2.00) > Millennial > Gen X > Boomer+ (1.94) |
| Income | Low income = highest loneliness |
| Relationship | In a relationship = highest loneliness among connection types |
| Smoking | Scores similar across all groups |
| Remote work | Work from home = highest loneliness (1.98) |
| Country | Variation exists across 27 EU countries |
 
---
 

In [36]:
loneliness.groupby('work_label')['overall loneliness index'].mean()

work_label
Employee/Self-Employed             1.962318
Housework                          1.983093
Other                              1.938413
Retired                            1.954403
Sick or Disabled                   1.982938
Studying                           2.020728
Unemployed-Actively looking        1.985141
Unemployed-Not actively looking    1.978792
Name: overall loneliness index, dtype: float64

In [37]:
loneliness.groupby('pet_label')['overall loneliness index'].mean()

pet_label
Has Pet    1.970859
No Pet     1.959133
Name: overall loneliness index, dtype: float64

In [38]:
loneliness.groupby('health_condition_label')['overall loneliness index'].mean()

health_condition_label
No     1.954230
Yes    1.983952
Name: overall loneliness index, dtype: float64

In [39]:
loneliness.groupby('child_lived_parents_label')['overall loneliness index'].mean()

child_lived_parents_label
No     1.969453
Yes    1.966555
Name: overall loneliness index, dtype: float64

In [40]:
loneliness.groupby('relationship_label')['overall loneliness index'].mean()

relationship_label
In a relationship        1.990376
Married                  1.952599
Separated or Divorces    1.962792
Single                   1.987294
Widowed                  1.970611
Name: overall loneliness index, dtype: float64

In [41]:
loneliness.groupby('health_condition_label')['overall loneliness index'].mean()

health_condition_label
No     1.954230
Yes    1.983952
Name: overall loneliness index, dtype: float64

In [42]:
loneliness.groupby('Smoker_label')['overall loneliness index'].mean()

Smoker_label
Heavy Smoker    1.965515
Non Smoker      1.965571
light Smoker    1.977169
Name: overall loneliness index, dtype: float64

In [43]:
loneliness.groupby('life_event_label')['overall loneliness index'].mean()

life_event_label
No     1.935081
Yes    1.984072
Name: overall loneliness index, dtype: float64

In [44]:
country_lonely=loneliness.groupby('country_label')['overall loneliness index'].mean()
country_lonely.sort_values(ascending=False)

country_label
Austria        2.023704
Sweden         2.022072
Greece         2.011241
Slovakia       2.001869
Netherlands    1.997538
Finland        1.993169
Germany        1.989329
Poland         1.987247
Denmark        1.986610
Cyprus         1.980472
Malta          1.977530
Ireland        1.976637
Estonia        1.972382
Luxembourg     1.969991
Italy          1.968834
Slovenia       1.968519
Portugal       1.967399
Czechia        1.958191
Croatia        1.953876
Belgium        1.950151
Hungary        1.946801
Spain          1.938576
Lithuania      1.928021
Latvia         1.920791
Bulgaria       1.919228
Romania        1.906769
France         1.897569
Name: overall loneliness index, dtype: float64

In [45]:
loneliness.groupby('remote_work_label')['overall loneliness index'].mean()

remote_work_label
Somewhere Else      1.954057
Work from Office    1.956603
Work from home      1.982002
Name: overall loneliness index, dtype: float64

In [46]:
sm_time_total = ['sm_time_a','sm_time_b','sm_time_c','sm_time_d']
loneliness.groupby('health_condition_label')[sm_time_total].mean()

,sm_time_a,sm_time_b,sm_time_c,sm_time_d
health_condition_label,,,,
No,3.803891,3.463396,2.662558,4.422241
Yes,3.854896,3.332952,2.706818,4.656238


In [47]:
loneliness.groupby('generation')[sm_time_total].mean()

C:\Users\Kanya\AppData\Local\Temp\ipykernel_32348\383108937.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  loneliness.groupby('generation')[sm_time_total].mean()


,sm_time_a,sm_time_b,sm_time_c,sm_time_d
generation,,,,
Gen Z,4.867503,4.542642,3.559105,4.506030
Millennial,4.114111,3.797640,2.999882,4.473982
Gen X,3.545099,3.056982,2.445615,4.534320
Boomer+,3.046882,2.529843,1.986632,4.469991


In [48]:
pd.crosstab(
    loneliness['health_condition_label'],
    loneliness['remote_work_label'],
    normalize='index'
) * 100

remote_work_label,Somewhere Else,Work from Office,Work from home
health_condition_label,,,
No,9.147833,69.366990,21.485178
Yes,7.503474,64.705882,27.790644


In [49]:
print(loneliness.groupby('remote_work_label')['overall loneliness index'].mean().sort_values(ascending=False))

remote_work_label
Work from home      1.982002
Work from Office    1.956603
Somewhere Else      1.954057
Name: overall loneliness index, dtype: float64


## Step 10 — Export Cleaned Dataset
 
Exporting the cleaned and enriched dataset for Power BI dashboard.
 
---

In [51]:
loneliness.to_csv('loneliness_clean.csv', index=False)

## Project Limitations
 
1. **Self-Reported Data** — Loneliness scores are based on how participants feel, not objective measurements. Responses may be influenced by social desirability bias.
 
2. **Correlation ≠ Causation** — Higher social media usage correlates with loneliness but does not prove SM causes loneliness. Lonely people may simply use SM more.
 
3. **European Bias** — Data covers only 27 European countries. Findings may not apply to Asia, Africa, or the Americas where social structures differ.
 
4. **Income Self-Reporting** — Income decile is self-reported and may not accurately reflect actual financial situation.
 
5. **Cross-Sectional Data** — This is a one-time survey snapshot. We cannot track how loneliness changes over time for the same individuals.
 
---

## Key Findings Summary
 
- **Gen Z** is the loneliest generation in Europe
- **Students** report higher loneliness than even unemployed individuals
- **Remote workers** feel lonelier than office workers, despite flexibility
- **Higher social media usage** correlates with higher loneliness
- **Using SM as a replacement** for face-to-face contact is associated with the highest loneliness scores
- **Health conditions and life events** both increase loneliness scores slightly
- **Low income** groups report higher loneliness than high income groups